# Advanced Pydantic Concepts

This notebook covers nested models, recursive models, serialization, and advanced configuration.

## 1. Nested Models

Pydantic models can be used as types inside other models, allowing for hierarchical data structures.

In [ ]:
from pydantic import BaseModel
from typing import List

class Address(BaseModel):
    street: str
    city: str
    zip_code: str

class User(BaseModel):
    name: str
    addresses: List[Address]

data = {
    "name": "Alice",
    "addresses": [
        {"street": "123 Main St", "city": "New York", "zip_code": "10001"},
        {"street": "456 Elm St", "city": "Boston", "zip_code": "02108"}
    ]
}

user = User(**data)
print(user.addresses[0].city)  # New York

## 2. Recursive Models (Self-Referencing)

Useful for tree-like structures (e.g., comments with replies). You need to use `model_rebuild()`.

In [ ]:
from typing import Optional

class Comment(BaseModel):
    id: int
    content: str
    replies: Optional[List['Comment']] = None

# Required for self-referencing models to resolve the type 'Comment'
Comment.model_rebuild()

comment = Comment(
    id=1, 
    content="Root comment", 
    replies=[
        Comment(id=2, content="Reply 1"),
        Comment(id=3, content="Reply 2", replies=[Comment(id=4, content="Nested reply")])
    ]
)
print(comment)

## 3. Serialization

- `model_dump()`: Converts the model to a Python dictionary.
- `model_dump_json()`: Converts the model to a JSON string.

In [ ]:
print("Dictionary:", user.model_dump())
print("JSON String:", user.model_dump_json())

## 4. Configuration (`ConfigDict`)

You can customize model behavior using `model_config`. A common use case is custom JSON encoding for types like `datetime`.

In [ ]:
from pydantic import ConfigDict
from datetime import datetime

class Event(BaseModel):
    name: str
    timestamp: datetime

    model_config = ConfigDict(
        json_encoders={datetime: lambda v: v.strftime("%Y-%m-%d %H:%M")}
    )

event = Event(name="Meeting", timestamp=datetime(2023, 10, 25, 14, 30))
print(event.model_dump_json())  # Timestamp will be formatted

## 5. Best Practices

- **Leaf Models First**: Define smaller, independent models (like `Address`) before using them in larger models (`User`).
- **Clear Naming**: Use descriptive names for models and fields.
- **Performance**: Be cautious with deeply nested or recursive models, as validation overhead increases.
- **Business Rules**: Always prioritize business logic in validators over simple type checks.